# IFRS S1/S2 + Commercial Banks Requirement Extraction — FINAL notebook

This notebook extracts structured disclosure requirements from the **three sources used in the bank reporting project**:

1. **IFRS S1** — General Requirements for Disclosure of Sustainability-related Financial Information
2. **IFRS S2** — Climate-related Disclosures
3. **IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks**

Final fixes included:

- Removed the old the duplicate enriched JSON output.
- Saves only one final requirements JSON: `ifrs_bank_requirements.json`.
- Keeps JSON-backed `requirements_kb.py` + `requirements_kb_data.json`.
- Normalises FN-CB metric codes canonically, e.g. `FN-CB-410a.2`.
- Prevents over-nested paragraph IDs from being treated as clean references.
- Adds suspicious paragraph QA.
- Adds smarter critical coverage checks across `paragraph`, `metric_type`, `related_paragraphs`, and `requirement_text`.
- Keeps all-section coverage checklists for generation and QA agents.

Important distinction:

- `core_standard` = IFRS S1 and IFRS S2 mandatory/core disclosure requirements
- `industry_guidance` = IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks

The Commercial Banks guidance accompanies IFRS S2 and supports application for commercial banks. It should not be presented as creating separate IFRS S2 requirements.


## 0. Folder structure expected

Create this structure before running the full extraction:

```text
project/
│
├── data/
│   ├── standards/
│   │   ├── ifrs_s1.pdf
│   │   ├── ifrs_s2.pdf
│   │   └── ifrs_s2_ibg_volume_16_commercial_banks.pdf
│   │
│   └── requirements/
│
└── notebooks/
    └── 01_extract_ifrs_requirements_S1_S2_CommercialBanks_AZURE_REST.ipynb
```

You can rename the PDF files, but if you do, update the `INPUT_PDFS` list in the configuration cell.

Recommended naming:

```text
ifrs_s1.pdf
ifrs_s2.pdf
ifrs_s2_ibg_volume_16_commercial_banks.pdf
```


In [1]:
# 1. Install packages
# Run this once in your notebook environment.

%pip install -q --upgrade pydantic pypdf pymupdf tenacity pandas python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 2. Imports

import os
import re
import json
import time
import random
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
from collections import defaultdict, Counter
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator, model_validator
from pypdf import PdfReader
from tenacity import retry, stop_after_attempt, wait_exponential
from dotenv import load_dotenv, find_dotenv

print("Imports loaded successfully.")


Imports loaded successfully.


In [3]:
# 3. Configuration

MODEL_LABEL = "Azure GPT-5.2 extraction deployment"

MAX_CHARS_PER_CHUNK = 12_000
MAX_CHUNKS_PER_DOCUMENT = None

# Higher limit to reduce truncation on long IFRS S2 metric paragraphs.
EXTRACTION_MAX_OUTPUT_TOKENS = 8000

BASE_DIR = Path(".")
STANDARDS_DIR = BASE_DIR / "gen_data" / "IFRS"
OUTPUT_DIR = BASE_DIR / "gen_data" / "requirements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Final single JSON output. The duplicate enriched output is removed.
OUTPUT_JSON = OUTPUT_DIR / "ifrs_bank_requirements.json"

# Fresh checkpoint for the final paragraph-aware extraction.
OUTPUT_RAW_JSONL = OUTPUT_DIR / "ifrs_bank_requirements_raw_checkpoint_final.jsonl"

OUTPUT_KB = OUTPUT_DIR / "requirements_kb.py"
OUTPUT_KB_DATA = OUTPUT_DIR / "requirements_kb_data.json"

CHECKLIST_DIR = OUTPUT_DIR / "coverage_checklists"
CHECKLIST_DIR.mkdir(parents=True, exist_ok=True)

REPORT_SECTION_VALUES = [
    "general_requirements",
    "governance",
    "strategy",
    "risk_management",
    "metrics_targets",
    "industry_metrics",
    "other",
]

APPLICABILITY_VALUES = [
    "commercial_banking",
    "asset_management",
    "insurance",
    "general_financial_sector",
    "not_financial_specific",
]

# Avoid paragraph references like 14(a)(i)(3)(iii)(3)(iv)(2).
MAX_PARAGRAPH_SUBLEVELS = 3

def first_existing_path(*candidates: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]

INPUT_PDFS = [
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s1.pdf",
            STANDARDS_DIR / "ifrs-s1-general-requirements.pdf",
        ),
        "standard": "S1",
        "source_doc": "IFRS S1 General Requirements",
        "source_authority": "core_standard",
    },
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s2.pdf",
            STANDARDS_DIR / "ifrs-s2-climate-related-disclosures.pdf",
        ),
        "standard": "S2",
        "source_doc": "IFRS S2 Climate-related Disclosures",
        "source_authority": "core_standard",
    },
    {
        "path": first_existing_path(
            STANDARDS_DIR / "ifrs_s2_ibg_volume_16_commercial_banks.pdf",
            STANDARDS_DIR / "ifrs-s2-ibg-volume-16-commercial-banks-part-b.pdf",
            STANDARDS_DIR / "ifrs-s2-ibg-volume-16-commercial-banks-part-b (1).pdf",
        ),
        "standard": "S2_IBG_CB",
        "source_doc": "IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks",
        "source_authority": "industry_guidance",
    },
]

print("Model/deployment label:", MODEL_LABEL)
print("Standards folder:", STANDARDS_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Checklist folder:", CHECKLIST_DIR.resolve())
print("Chunk size:", MAX_CHARS_PER_CHUNK)
print("Max output tokens:", EXTRACTION_MAX_OUTPUT_TOKENS)
print("Checkpoint file:", OUTPUT_RAW_JSONL.resolve())

print("\nConfigured source documents:")
for doc in INPUT_PDFS:
    print(f"- {doc['standard']}: {doc['source_doc']} -> {doc['path']}")


Model/deployment label: Azure GPT-5.2 extraction deployment
Standards folder: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS
Output folder: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements
Checklist folder: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists
Chunk size: 12000
Max output tokens: 8000
Checkpoint file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\ifrs_bank_requirements_raw_checkpoint_final.jsonl

Configured source documents:
- S1: IFRS S1 General Requirements -> gen_data\IFRS\ifrs_s1.pdf
- S2: IFRS S2 Climate-related Disclosures -> gen_data\IFRS\ifrs_s2.pdf
- S2_IBG_CB: IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks -> gen_data\IFRS\ifrs-s2-ibg-volume-16-commercial-banks-part-b (1).pdf


In [4]:
# 4. Check that your PDFs exist

missing = []

for item in INPUT_PDFS:
    path = item["path"]
    if path.exists():
        print(f"FOUND: {path}")
    else:
        print(f"MISSING: {path}")
        missing.append(path)

if missing:
    print("\nSome PDFs are missing. Add them to data/standards/ or update INPUT_PDFS.")
else:
    print("\nAll PDFs found.")


FOUND: gen_data\IFRS\ifrs_s1.pdf
FOUND: gen_data\IFRS\ifrs_s2.pdf
FOUND: gen_data\IFRS\ifrs-s2-ibg-volume-16-commercial-banks-part-b (1).pdf

All PDFs found.


In [5]:
# 5. Azure OpenAI REST setup
# This cell matches the logic style of your working governance notebook.

# Safety defaults in case this cell is run before the configuration cell.
# The config cell values still take priority when already defined.
EXTRACTION_MAX_OUTPUT_TOKENS = globals().get("EXTRACTION_MAX_OUTPUT_TOKENS", 8000)
MAX_CHARS_PER_CHUNK = globals().get("MAX_CHARS_PER_CHUNK", 12000)
MAX_CHUNKS_PER_DOCUMENT = globals().get("MAX_CHUNKS_PER_DOCUMENT", None)

print("Runtime defaults checked:")
print("EXTRACTION_MAX_OUTPUT_TOKENS =", EXTRACTION_MAX_OUTPUT_TOKENS)
print("MAX_CHARS_PER_CHUNK =", MAX_CHARS_PER_CHUNK)

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_EXTRACTOR_API_KEY = (
    os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY")
    or _shared_key_fallback
)

AZURE_OPENAI_EXTRACTOR_URL = _clean_url(
    os.getenv("AZURE_OPENAI_EXTRACTOR_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)


def validate_extractor_config() -> None:
    missing = []

    if not AZURE_OPENAI_EXTRACTOR_API_KEY:
        missing.append("AZURE_OPENAI_EXTRACTOR_API_KEY or AZURE_OPENAI_API_KEY")

    if not AZURE_OPENAI_EXTRACTOR_URL:
        missing.append("AZURE_OPENAI_EXTRACTOR_URL or AZURE_OPENAI_JUDGE_URL")

    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "extractor_key_loaded": bool(AZURE_OPENAI_EXTRACTOR_API_KEY),
            "judge_key_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_API_KEY")),
            "extractor_url_loaded": bool(os.getenv("AZURE_OPENAI_EXTRACTOR_URL")),
            "judge_url_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_URL")),
            "chat_url_loaded": bool(os.getenv("AZURE_OPENAI_CHAT_URL")),
        }

        raise ValueError(
            "Missing Azure extractor configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags, keys are never printed:\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nExpected .env example:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_EXTRACTOR_URL=<full GPT-5.2 chat-completions deployment URL>\n\n"
              "Alternative using your existing governance setup:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 chat-completions deployment URL>"
        )

    if not AZURE_OPENAI_EXTRACTOR_URL.startswith("https://"):
        raise ValueError(
            "AZURE_OPENAI_EXTRACTOR_URL must be a full HTTPS Azure deployment URL. "
            f"Current value: {AZURE_OPENAI_EXTRACTOR_URL!r}"
        )

    if "/chat/completions" not in AZURE_OPENAI_EXTRACTOR_URL:
        print("WARNING: The extractor URL does not contain '/chat/completions'.")
        print("Expected full URL format:")
        print("https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=<version>")


validate_extractor_config()

print("Azure OpenAI extractor configuration loaded.")
print("Extractor endpoint:", AZURE_OPENAI_EXTRACTOR_URL[:110] + "...")
print("API key loaded:", bool(AZURE_OPENAI_EXTRACTOR_API_KEY))


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = 0.0,
    use_max_completion_tokens: bool = True,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 5,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 429/500/502/503/504 and connection errors.
    - For 429, honours Retry-After when provided.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway rejects it, retries using `max_tokens`.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )

    token_fields = [preferred_field]

    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        # v4: explicit deterministic extraction.
        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {429, 500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if transient and attempt < max_attempts:
                    if exc.code == 429:
                        retry_after_raw = exc.headers.get("Retry-After")
                        try:
                            wait = float(retry_after_raw) if retry_after_raw else 10.0
                        except ValueError:
                            wait = 10.0

                        wait = min(max(wait, 1.0), 60.0)
                        print(
                            f"{request_label}: rate limited (429); "
                            f"retrying attempt {attempt + 1}/{max_attempts} "
                            f"in {wait:.1f}s..."
                        )
                    else:
                        wait = min(2 ** (attempt - 1) + random.random(), 12)
                        print(
                            f"{request_label}: server error {exc.code}; "
                            f"retrying attempt {attempt + 1}/{max_attempts} "
                            f"in {wait:.1f}s..."
                        )

                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_extractor_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 extractor with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Requires a JSON object with a top-level `requirements` array.

    On invalid/truncated JSON:
    - Sends the returned content back to the same Azure endpoint for JSON repair.
    """

    data = _azure_chat_completion(
        url=AZURE_OPENAI_EXTRACTOR_URL,
        api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=EXTRACTION_MAX_OUTPUT_TOKENS,
        use_max_completion_tokens=True,
        temperature=0.0,
        json_mode=True,
        request_label="GPT-5.2 IFRS extractor",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("Extractor returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "The object must contain a top-level key named requirements. "
            "The value must be an array. "
            "Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated extractor output into one complete valid JSON object.

Required schema:
{{
  "requirements": [
    {{
      "standard": "S1 or S2 or S2_IBG_CB",
      "source_doc": "string",
      "source_authority": "core_standard or industry_guidance",
      "paragraph": "string",
      "report_sections": ["risk_management"],
      "applicability": "commercial_banking",
      "section": "general_requirements or governance or strategy or risk_management or metrics_targets or industry_metrics or other",
      "obligation_type": "shall or should or may",
      "requirement_text": "string",
      "applies_to_banks": true,
      "related_paragraphs": [],
      "metric_type": null,
      "page_start": 1,
      "page_end": 1
    }}
  ]
}}

Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_EXTRACTOR_URL,
            api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=EXTRACTION_MAX_OUTPUT_TOKENS,
            use_max_completion_tokens=True,
            temperature=0.0,
            json_mode=True,
            request_label="GPT-5.2 IFRS extractor JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "Extractor failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


print("Azure REST helper functions ready.")


Runtime defaults checked:
EXTRACTION_MAX_OUTPUT_TOKENS = 8000
MAX_CHARS_PER_CHUNK = 12000
Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Azure OpenAI extractor configuration loaded.
Extractor endpoint: https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/completions...
API key loaded: True
Azure REST helper functions ready.


In [6]:
# 5B. Optional smoke test: verify the Azure extractor endpoint responds

# This cell calls Azure once with a tiny JSON-mode request.
# Run this before the real extraction if you want to confirm the endpoint works.

smoke_result = call_extractor_llm_json(
    system_prompt="You return valid JSON only.",
    user_prompt='Return exactly this JSON object: {"requirements": []}'
)

print(smoke_result)


{'requirements': []}


In [7]:
# 6. Structured output schema

ReportSection = Literal[
    "general_requirements",
    "governance",
    "strategy",
    "risk_management",
    "metrics_targets",
    "industry_metrics",
    "other",
]

Applicability = Literal[
    "commercial_banking",
    "asset_management",
    "insurance",
    "general_financial_sector",
    "not_financial_specific",
]


def canonical_fn_cb_code(value: str) -> str:
    """Canonicalise FN-CB codes without incorrectly uppercasing topic letters."""
    value = str(value or "").strip()

    match = re.search(
        r"FN-CB-(\d{3})([A-Za-z]?)[.](\d+|[A-Za-z])",
        value,
        flags=re.IGNORECASE,
    )

    if not match:
        return value

    number, topic_letter, suffix = match.groups()

    if topic_letter and suffix.isdigit():
        return f"FN-CB-{number}{topic_letter.lower()}.{suffix}"

    if not topic_letter and suffix.isalpha():
        return f"FN-CB-{number}.{suffix.upper()}"

    if topic_letter and suffix.isalpha():
        return f"FN-CB-{number}{topic_letter.lower()}.{suffix.upper()}"

    return f"FN-CB-{number}.{suffix}"


def normalise_paragraph_id(value: str) -> str:
    value = str(value or "").strip()

    if not value:
        return "unknown"

    if value.lower().startswith("unknown_page_"):
        return "unknown"

    if re.match(r"^note[-\s]+to[-\s]+FN-CB-", value, flags=re.IGNORECASE):
        value = re.sub(r"\s+", "-", value.strip(), flags=re.IGNORECASE)
        value = re.sub(r"-+", "-", value)
        return value.upper().replace("FN-CB-410A", "FN-CB-410a")

    if re.search(r"FN-CB-", value, flags=re.IGNORECASE):
        return canonical_fn_cb_code(value)

    # Avoid accidental full-text paragraph values.
    if len(value) > 45 or " " in value:
        return "unknown"

    base_match = re.match(
        r"^([A-Z]?\d{1,3}[A-Z]?|\d{1,3}(?:\.\d+)+)",
        value,
        flags=re.IGNORECASE,
    )

    if base_match:
        base = base_match.group(1)
        groups = re.findall(r"\(([a-z0-9]+)\)", value, flags=re.IGNORECASE)

        clean_groups = []
        roman_values = {
            "i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x",
            "xi", "xii", "xiii", "xiv", "xv"
        }

        for g in groups:
            g_l = g.lower()

            if not (g_l.isdigit() or re.fullmatch(r"[a-z]", g_l) or g_l in roman_values):
                break

            clean_groups.append(g_l)

            if len(clean_groups) >= globals().get("MAX_PARAGRAPH_SUBLEVELS", 3):
                break

        base_out = base.upper() if base[:1].isalpha() else base
        return base_out + "".join(f"({g})" for g in clean_groups)

    return value


def is_suspicious_paragraph_id(value: str) -> bool:
    value = str(value or "").strip()

    if not value or value == "unknown":
        return True

    if "gaap" in value.lower():
        return True

    if len(value) > 45:
        return True

    if value.count("(") > globals().get("MAX_PARAGRAPH_SUBLEVELS", 3):
        return True

    if re.match(r"^NOTE-TO-FN-CB-", value, flags=re.IGNORECASE):
        return False

    if re.match(r"^FN-CB-\d{3}[a-z]\.\d$", value):
        return False

    if re.match(r"^FN-CB-\d{3}\.[A-Z]$", value):
        return False

    if re.match(
        r"^([A-Z]\d+[A-Z]?|\d{1,3}[A-Z]?|\d{1,3}(?:\.\d+)+)(?:\([a-z0-9]+\))*$",
        value,
        flags=re.IGNORECASE,
    ):
        return False

    return True


class IFRSRequirement(BaseModel):
    model_config = ConfigDict(extra="forbid")

    standard: Literal["S1", "S2", "S2_IBG_CB"]
    source_doc: str
    source_authority: Literal["core_standard", "industry_guidance"]

    paragraph: str
    section: ReportSection
    report_sections: list[ReportSection]

    obligation_type: Literal["shall", "should", "may"]
    requirement_text: str

    applies_to_banks: bool
    applicability: Applicability

    related_paragraphs: list[str]
    metric_type: str | None

    page_start: int
    page_end: int

    @field_validator("paragraph")
    @classmethod
    def normalise_paragraph(cls, value: str) -> str:
        return normalise_paragraph_id(value)

    @field_validator("report_sections")
    @classmethod
    def normalise_report_sections(cls, value: list[str]) -> list[str]:
        cleaned = []

        for section in value or []:
            if section in REPORT_SECTION_VALUES and section not in cleaned:
                cleaned.append(section)

        return cleaned or ["other"]

    @field_validator("related_paragraphs")
    @classmethod
    def normalise_related_paragraphs(cls, value: list[str]) -> list[str]:
        cleaned = []

        for item in value or []:
            item = normalise_paragraph_id(str(item).strip())

            if not item or item == "unknown":
                continue

            if item not in cleaned:
                cleaned.append(item)

        return cleaned

    @field_validator("metric_type")
    @classmethod
    def normalise_metric_type(cls, value: str | None) -> str | None:
        if value is None:
            return None

        value = str(value).strip()

        if not value or value.lower() in {"none", "null", "n/a", "na"}:
            return None

        if re.search(r"FN-CB-", value, flags=re.IGNORECASE):
            return canonical_fn_cb_code(value)

        return value

    @model_validator(mode="after")
    def sync_applies_to_banks(self) -> "IFRSRequirement":
        self.applies_to_banks = self.applicability in (
            "commercial_banking",
            "general_financial_sector",
        )
        return self


class RequirementExtractionResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    requirements: list[IFRSRequirement]


print("Pydantic schemas and paragraph QA helpers ready.")


Pydantic schemas and paragraph QA helpers ready.


In [8]:
# 7. PDF parsing helpers + generic paragraph-aware parser

def clean_text(text: str) -> str:
    """Basic PDF text cleanup while preserving useful line breaks."""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


try:
    import fitz  # PyMuPDF
    _HAS_FITZ = True
except Exception:
    _HAS_FITZ = False

# A token that looks like an IFRS paragraph number printed in the margin column
# (body paragraphs: 1..999 with optional trailing letter; appendix: B62, C1, etc.).
_MARGIN_NUM_RE = re.compile(r"^(?:[A-Z]?\d{1,3}[A-Za-z]?)$")


def _looks_like_margin_number(tok: str) -> bool:
    tok = str(tok).strip().rstrip(".")
    return bool(tok) and bool(_MARGIN_NUM_RE.match(tok))


def _reconstruct_page_text_with_margin_numbers(words, page_width, page_height) -> str:
    """Bind margin paragraph-numbers to the body line they sit beside.

    IFRS standards print paragraph numbers in a narrow margin column; a plain
    text dump detaches them from their text. Using word coordinates we identify
    the margin-number column and PREPEND each number to the body line at the same
    vertical position, so the existing line-start marker detection works correctly.

    `words` is PyMuPDF's get_text("words"): (x0, y0, x1, y1, text, block, line, word_no).
    """
    if not words:
        return ""

    # 1. Drop header/footer bands (running titles, footer page numbers).
    top_band = page_height * 0.06
    bot_band = page_height * 0.94
    ws = [w for w in words if top_band <= (w[1] + w[3]) / 2.0 <= bot_band] or list(words)

    # 2. Body column edges = median left/right of multi-word (text) lines.
    line_words = defaultdict(list)
    for w in ws:
        line_words[(w[5], w[6])].append(w)
    lefts, rights = [], []
    for lw in line_words.values():
        if len(lw) >= 2:
            lefts.append(min(x[0] for x in lw))
            rights.append(max(x[2] for x in lw))
    lefts.sort(); rights.sort()
    body_left = lefts[len(lefts) // 2] if lefts else min((w[0] for w in ws), default=0.0)
    body_right = rights[len(rights) // 2] if rights else max((w[2] for w in ws), default=page_width)

    # 3. Separate margin-number tokens (numeric/appendix codes sitting OUTSIDE the
    #    body column on either side) from body words.
    gap = 6.0
    margins, body_words = [], []
    for w in ws:
        x0, y0, x1, y1, txt = w[0], w[1], w[2], w[3], w[4]
        is_margin = _looks_like_margin_number(txt) and (x1 <= body_left - gap or x0 >= body_right + gap)
        (margins if is_margin else body_words).append(w)
    margins = [((w[1] + w[3]) / 2.0, str(w[4]).strip().rstrip(".")) for w in margins]

    # 4. Rebuild body lines in reading order.
    body_lines = []
    bl = defaultdict(list)
    for w in body_words:
        bl[(w[5], w[6])].append(w)
    for lw in bl.values():
        lw.sort(key=lambda w: w[0])
        y_mid = sum((w[1] + w[3]) / 2.0 for w in lw) / len(lw)
        body_lines.append([y_mid, " ".join(str(w[4]) for w in lw)])
    body_lines.sort(key=lambda r: r[0])

    # 5. Bind each margin number to the nearest unused body line (its paragraph's
    #    first line), within ~one line height. Strays with no match are dropped.
    used = set()
    line_h = max(8.0, page_height * 0.018)
    for y_mid, num in sorted(margins):
        best, best_d = None, 1e9
        for idx, (ly, _t) in enumerate(body_lines):
            if idx in used:
                continue
            d = abs(ly - y_mid)
            if d < best_d:
                best_d, best = d, idx
        if best is not None and best_d <= line_h * 1.2:
            used.add(best)
            body_lines[best][1] = f"{num} {body_lines[best][1]}"

    return "\n".join(t for _y, t in body_lines)


def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """Extract text page by page, binding margin paragraph-numbers to their text
    via word coordinates (PyMuPDF). Falls back to pypdf if PyMuPDF is unavailable
    (legacy behaviour: margin numbers will not be bound)."""
    pages = []

    if _HAS_FITZ:
        doc = fitz.open(str(pdf_path))
        try:
            for i, page in enumerate(doc, start=1):
                words = page.get_text("words")
                rect = page.rect
                text = clean_text(
                    _reconstruct_page_text_with_margin_numbers(words, rect.width, rect.height)
                )
                if text:
                    pages.append({"page": i, "text": text})
        finally:
            doc.close()
        if pages:
            return pages

    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            pages.append({"page": i, "text": text})
    return pages


PARAGRAPH_MARKER_PATTERNS = [
    re.compile(r"^\s*(FN-CB-\d{3}[a-z]\.\d|FN-CB-\d{3}\.[A-Z])\.?\s*(.*)$", re.IGNORECASE),
    re.compile(r"^\s*(Note\s+to\s+FN-CB-\d{3}[a-z]\.\d)\s*(.*)$", re.IGNORECASE),
    re.compile(r"^\s*([A-Z]\d+[A-Z]?(?:\([a-z0-9]+\))*)\.?\s+(.*)$", re.IGNORECASE),
    re.compile(r"^\s*(\d{1,3}[A-Z]?(?:\([a-z0-9]+\))*|\d{1,3}(?:\.\d+)+)\.?\s+(.*)$", re.IGNORECASE),
]

ORPHAN_SUBITEM_PATTERN = re.compile(r"^\s*\(([a-z0-9]+)\)\s+(.*)$", re.IGNORECASE)


def _paren_groups(paragraph_id: str) -> list[str]:
    return re.findall(r"\(([a-z0-9]+)\)", str(paragraph_id), flags=re.IGNORECASE)


def _token_type(token: str) -> str:
    token = str(token).lower()

    roman_values = {
        "i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x",
        "xi", "xii", "xiii", "xiv", "xv"
    }

    if token.isdigit():
        return "number"

    if token in roman_values:
        return "roman"

    if re.fullmatch(r"[a-z]", token):
        return "letter"

    return "other"


def append_or_replace_orphan_subitem(parent_id: str, orphan_token: str) -> str:
    parent_id = normalise_paragraph_id(str(parent_id or "unknown"))
    orphan_token = str(orphan_token).lower()

    if parent_id.startswith("unknown"):
        return "unknown"

    groups = _paren_groups(parent_id)

    if len(groups) >= MAX_PARAGRAPH_SUBLEVELS:
        return parent_id

    if not groups:
        return normalise_paragraph_id(f"{parent_id}({orphan_token})")

    last = groups[-1]
    last_type = _token_type(last)
    orphan_type = _token_type(orphan_token)

    if orphan_type == "other":
        return parent_id

    if last_type == orphan_type:
        base = re.sub(r"\([a-z0-9]+\)$", "", parent_id, flags=re.IGNORECASE)
        return normalise_paragraph_id(f"{base}({orphan_token})")

    return normalise_paragraph_id(f"{parent_id}({orphan_token})")


def detect_paragraph_marker(line: str) -> tuple[str, str] | None:
    """Detect paragraph or metric marker at the beginning of a line."""
    line = str(line).strip()

    if not line:
        return None

    for pattern in PARAGRAPH_MARKER_PATTERNS:
        match = pattern.match(line)
        if not match:
            continue

        marker = normalise_paragraph_id(match.group(1).strip())
        rest = match.group(2).strip()

        if marker.isdigit():
            marker_int = int(marker)

            if marker_int >= 1900:
                return None

            if re.match(r"^(january|february|march|april|may|june|july|august|september|october|november|december)\b", rest, re.I):
                return None

            if len(rest) < 8 and "ifrs" in rest.lower():
                return None

        return marker, rest

    return None


def extract_paragraph_units_from_pages(
    pages: list[dict],
    source_doc: str,
    standard: str,
    source_authority: str,
) -> list[dict]:
    units = []
    current = None

    def flush_current():
        nonlocal current
        if current and current.get("text", "").strip():
            current["paragraph"] = normalise_paragraph_id(current["paragraph"])
            current["text"] = re.sub(r"\s+", " ", current["text"]).strip()
            units.append(current)
        current = None

    for page in pages:
        page_no = page["page"]
        lines = [ln.strip() for ln in page["text"].splitlines() if ln.strip()]

        for line in lines:
            orphan = ORPHAN_SUBITEM_PATTERN.match(line)

            if orphan and current is not None:
                token, rest = orphan.group(1), orphan.group(2)
                new_marker = append_or_replace_orphan_subitem(current["paragraph"], token)

                # Avoid fake ultra-nested paragraph IDs; keep such text in the current unit.
                if new_marker == current["paragraph"] or is_suspicious_paragraph_id(new_marker):
                    current["text"] += " " + line
                    current["page_end"] = page_no
                    continue

                flush_current()

                current = {
                    "standard": standard,
                    "source_doc": source_doc,
                    "source_authority": source_authority,
                    "paragraph": new_marker,
                    "page_start": page_no,
                    "page_end": page_no,
                    "text": rest,
                }
                continue

            detected = detect_paragraph_marker(line)

            if detected:
                marker, rest = detected
                flush_current()

                current = {
                    "standard": standard,
                    "source_doc": source_doc,
                    "source_authority": source_authority,
                    "paragraph": marker,
                    "page_start": page_no,
                    "page_end": page_no,
                    "text": rest if rest else line,
                }

            else:
                if current is None:
                    current = {
                        "standard": standard,
                        "source_doc": source_doc,
                        "source_authority": source_authority,
                        "paragraph": f"unknown_page_{page_no}",
                        "page_start": page_no,
                        "page_end": page_no,
                        "text": line,
                    }
                else:
                    current["text"] += " " + line
                    current["page_end"] = page_no

    flush_current()

    return units


def make_chunks_from_units(
    units: list[dict],
    max_chars: int = MAX_CHARS_PER_CHUNK,
) -> list[dict]:
    chunks = []
    current_blocks = []
    current_pages = []
    current_paragraphs = []
    current_len = 0

    def build_block(unit: dict) -> str:
        return (
            f"\n\n[PARAGRAPH_ID: {unit['paragraph']} | "
            f"PAGES: {unit['page_start']}-{unit['page_end']}]\n"
            f"{unit['text']}"
        )

    for unit in units:
        block = build_block(unit)

        if current_blocks and current_len + len(block) > max_chars:
            chunks.append({
                "text": "\n".join(current_blocks),
                "page_start": min(current_pages),
                "page_end": max(current_pages),
                "char_count": current_len,
                "unit_count": len(current_blocks),
                "paragraphs": list(current_paragraphs),
            })

            current_blocks = []
            current_pages = []
            current_paragraphs = []
            current_len = 0

        current_blocks.append(block)
        current_pages.extend([unit["page_start"], unit["page_end"]])
        current_paragraphs.append(unit["paragraph"])
        current_len += len(block)

    if current_blocks:
        chunks.append({
            "text": "\n".join(current_blocks),
            "page_start": min(current_pages),
            "page_end": max(current_pages),
            "char_count": current_len,
            "unit_count": len(current_blocks),
            "paragraphs": list(current_paragraphs),
        })

    return chunks


def make_chunks(
    pages: list[dict],
    max_chars: int = MAX_CHARS_PER_CHUNK,
    source_doc: str = "",
    standard: str = "",
    source_authority: str = "",
) -> list[dict]:
    units = extract_paragraph_units_from_pages(
        pages=pages,
        source_doc=source_doc,
        standard=standard,
        source_authority=source_authority,
    )

    return make_chunks_from_units(units, max_chars=max_chars)


print("PDF parsing and paragraph-aware chunking helpers ready.")


PDF parsing and paragraph-aware chunking helpers ready.


In [9]:
# 8. Test PDF parsing and paragraph-aware unit extraction on the first available PDF

available_pdfs = [x for x in INPUT_PDFS if x["path"].exists()]

if not available_pdfs:
    raise FileNotFoundError("No PDFs found. Add PDFs to data/standards/ first.")

test_pdf = available_pdfs[0]
pages = extract_pdf_pages(test_pdf["path"])

units = extract_paragraph_units_from_pages(
    pages=pages,
    source_doc=test_pdf["source_doc"],
    standard=test_pdf["standard"],
    source_authority=test_pdf["source_authority"],
)

chunks = make_chunks_from_units(units)

print("Test document:", test_pdf["source_doc"])
print("Pages extracted:", len(pages))
print("Paragraph-aware units created:", len(units))
print("Chunks created:", len(chunks))

if pages:
    print("\nFirst page preview:")
    print(pages[0]["text"][:1200])

if units:
    print("\nFirst 10 paragraph-aware units:")
    for u in units[:10]:
        print(f"- {u['paragraph']} | pages {u['page_start']}-{u['page_end']} | {u['text'][:180]}...")


Test document: IFRS S1 General Requirements
Pages extracted: 48
Paragraph-aware units created: 244
Chunks created: 10

First page preview:
June 2023
IFRS S1
® Sustainability Disclosure Standard
IFRS
General Requirements for Disclosure of
Sustainability-related Financial Information

First 10 paragraph-aware units:
- unknown | pages 1-4 | June 2023 IFRS S1 ® Sustainability Disclosure Standard IFRS General Requirements for Disclosure of Sustainability-related Financial Information IFRS S1 General Requirements for Dis...
- S1 | pages 4-4 | GENERAL REQUIREMENTS FOR DISCLOSURE OF SUSTAINABILITY-RELATED FINANCIAL INFORMATION...
- 1 | pages 4-4 | OBJECTIVE...
- 5 | pages 4-4 | SCOPE...
- 10 | pages 4-4 | CONCEPTUAL FOUNDATIONS...
- 11 | pages 4-4 | Fair presentation...
- 17 | pages 4-4 | Materiality...
- 20 | pages 4-4 | Reporting entity...
- 21 | pages 4-4 | Connected information...
- 25 | pages 4-4 | CORE CONTENT...


In [10]:
# 9. Preview paragraph-aware chunks before sending to GPT

chunk_preview_rows = []

for pdf_info in available_pdfs:
    pages = extract_pdf_pages(pdf_info["path"])

    units = extract_paragraph_units_from_pages(
        pages=pages,
        source_doc=pdf_info["source_doc"],
        standard=pdf_info["standard"],
        source_authority=pdf_info["source_authority"],
    )

    chunks = make_chunks_from_units(units)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_preview_rows.append({
            "source_doc": pdf_info["source_doc"],
            "standard": pdf_info["standard"],
            "source_authority": pdf_info["source_authority"],
            "chunk_index": idx,
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"],
            "char_count": chunk["char_count"],
            "unit_count": chunk["unit_count"],
            "paragraphs_preview": ", ".join(chunk["paragraphs"][:12]),
            "text_preview": chunk["text"][:300].replace("\n", " "),
        })

chunk_df = pd.DataFrame(chunk_preview_rows)
chunk_df.head(20)


,source_doc,standard,source_authority,chunk_index,page_start,page_end,char_count,unit_count,paragraphs_preview,text_preview
0,IFRS S1 General Requirements,S1,core_standard,1,1,9,11974,47,"unknown, S1, 1, 5, 10, 11, 17, 20, 21, 25, 26, 28",[PARAGRAPH_ID: unknown | PAGES: 1-4] June 20...
1,IFRS S1 General Requirements,S1,core_standard,2,9,13,11686,20,"20, 21, 22, 8, 23, 24, 25, 26, 27, S1, 28, 29",[PARAGRAPH_ID: 20 | PAGES: 9-9] An entity’s ...
2,IFRS S1 General Requirements,S1,core_standard,3,13,18,11957,27,"12, 36, 37, 38, 39, 40, S1, 41, 42, 43, 44, 14",[PARAGRAPH_ID: 12 | PAGES: 13-14] © IFRS Fou...
3,IFRS S1 General Requirements,S1,core_standard,4,18,22,11584,25,"58, 59, 60, 61, 62, 63, 18, 64, 65, 66, 12, 67",[PARAGRAPH_ID: 58 | PAGES: 18-19] In making ...
4,IFRS S1 General Requirements,S1,core_standard,5,22,27,11678,15,"80, 81, 82, 83, 84, 85, 86, 22, S1, 24, B1, B2",[PARAGRAPH_ID: 80 | PAGES: 22-22] The requir...
5,IFRS S1 General Requirements,S1,core_standard,6,27,31,11453,23,"B6, B7, B8, 26, B9, B10, B11, B12, B13, B14, B...",[PARAGRAPH_ID: B6 | PAGES: 27-27] An entity ...
6,IFRS S1 General Requirements,S1,core_standard,7,31,34,10757,19,"B27, B28, B29, 30, B30, B31, B32, B33, B34, B3...",[PARAGRAPH_ID: B27 | PAGES: 31-31] An entity...
7,IFRS S1 General Requirements,S1,core_standard,8,34,39,11919,25,"B44, B45, B46, B47, B48, 34, B49, B50, B51, B5...",[PARAGRAPH_ID: B44 | PAGES: 34-35] Other exa...
8,IFRS S1 General Requirements,S1,core_standard,9,39,43,11840,28,"D5, 38, D6, D7, D8, D9, D10, D11, D12, D13, D1...",[PARAGRAPH_ID: D5 | PAGES: 39-39] Sustainabi...
9,IFRS S1 General Requirements,S1,core_standard,10,43,48,5391,15,"D31, 42, D32, D33, E1, E2, E3, E4, E5, E6, 44, S1",[PARAGRAPH_ID: D31 | PAGES: 43-43] The compl...


In [11]:
# 10. Prompts

SYSTEM_PROMPT = """
You are an IFRS S1/S2 and banking disclosure requirement extraction specialist.

Your job is to extract structured disclosure requirements from three source types:

1. IFRS S1 core standard
2. IFRS S2 core standard
3. IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks

Important source authority distinction:
- For S1 and S2, extract mandatory disclosure requirements as core standard requirements.
- For S2_IBG_CB, extract the Commercial Banks industry-based guidance as industry guidance.
  This guidance accompanies IFRS S2 and suggests ways to apply IFRS S2 for commercial banks.
  It does not create additional IFRS S2 requirements.

The input text is already split into paragraph-aware units.
Each unit begins with a header like:
[PARAGRAPH_ID: 27(a)(ii) | PAGES: 10-10]

Use the PARAGRAPH_ID from that header as the paragraph value.
Do not guess paragraph numbers.
Do not infer paragraph numbers from semantic keywords.
If the paragraph header is unknown_page_X and no clearer code appears in the unit text, use "unknown".

Critical extraction rule for sub-items:
- When a paragraph has labelled sub-items, for example 29(a), 29(b), 29(c), or 29(a)(i), extract each labelled sub-item as a separate requirement record.
- Do not collapse multiple labelled sub-items into one requirement_text.
- If a parent paragraph contains only an objective/introduction and the sub-items contain the actual disclosure obligations, extract the sub-items separately and omit the parent unless the parent itself creates a disclosure obligation.

Critical extraction rule for FN-CB notes:
- "Note to FN-CB-*" paragraphs may contain additional disclosure obligations.
- If a note uses "shall", extract it as a separate requirement with obligation_type="shall".
- Use a paragraph value such as "NOTE-TO-FN-CB-410a.2" when that note identifier is visible.

Extract only actual disclosure obligations, application guidance, technical protocols, metrics, or activity metrics that affect disclosure.

A requirement or guidance item is usually indicated by wording such as:
- shall disclose
- shall include
- shall describe
- shall provide
- shall explain
- is required to disclose
- should disclose
- may disclose
- metric
- activity metric
- technical protocol
- scope of disclosure

Do not invent requirements.
Do not paraphrase the paragraph text.
The requirement_text must be copied from the provided text as closely as possible.

Classify each item into one primary section:
- general_requirements
- governance
- strategy
- risk_management
- metrics_targets
- industry_metrics
- other

Also provide report_sections, which can include multiple sections that the item supports.

Classification guidance:
- Use industry_metrics for Commercial Banks FN-CB metrics and activity metrics.
- Use risk_management for credit analysis, ESG integration in lending, portfolio risk, scenario analysis, risk identification, and credit exposure concentration.
- Use metrics_targets for climate metrics such as GHG emissions, Scope 1/2/3, targets, capital deployment, physical risk, transition risk, climate opportunities, financed emissions metrics, and activity metrics.
- Use governance for board oversight, management roles, controls, oversight and accountability.
- Use strategy for business model, value chain, financial effects, resilience, transition plans, and climate-related opportunities.
- Use general_requirements for materiality, reporting entity, timing, location, comparative information, judgement, estimation uncertainty and general presentation requirements.

Applicability values:
- commercial_banking: commercial banks, lending, loans, credit analysis, project finance, borrowers, collateral, credit exposure, commercial banking financed emissions
- asset_management: asset management, AUM, investment portfolios or managed funds only
- insurance: insurance underwriting or insurance activities only
- general_financial_sector: financial institutions or finance-sector language that is not limited to one activity
- not_financial_specific: general IFRS requirement that applies to all entities

Applicability guidance:
- IFRS S1 requirements that are foundational to all sustainability disclosures should usually be applicability="not_financial_specific".
- IFRS S1/S2 requirements that explicitly mention financial institutions or financial-sector activities should use "general_financial_sector" or the specific activity.
- Set applies_to_banks to true only when applicability is commercial_banking or general_financial_sector.
- Set applies_to_banks to false when applicability is asset_management, insurance or not_financial_specific.

For all S2_IBG_CB items:
- applicability must be commercial_banking
- applies_to_banks must be true
- report_sections should normally include industry_metrics and at least one of risk_management, strategy, metrics_targets or governance depending on the item.
- metric_type should be the exact FN-CB metric code when visible, for example "FN-CB-410a.2".

You must return a valid JSON object only.
The JSON object must have exactly this top-level structure:
{
  "requirements": [...]
}

Do not return a bare array.
Do not include markdown fences.
Do not include commentary.
""".strip()


def build_user_prompt(
    text_chunk: str,
    standard: str,
    source_doc: str,
    source_authority: str,
    page_start: int,
    page_end: int,
) -> str:
    return f"""
STANDARD: {standard}
SOURCE DOCUMENT: {source_doc}
SOURCE AUTHORITY: {source_authority}
PAGE RANGE: {page_start}-{page_end}

The text below contains paragraph-aware units.
Each unit starts with:
[PARAGRAPH_ID: ... | PAGES: ...]

Extract every relevant disclosure requirement, guidance item, metric, activity metric, or technical protocol from the units below.

For every extracted item:
- standard must be "{standard}"
- source_doc must be "{source_doc}"
- source_authority must be "{source_authority}"
- page_start and page_end should match the unit page header where possible
- paragraph must come from the unit's PARAGRAPH_ID header, for example:
  - "29"
  - "29A"
  - "29(a)"
  - "29(a)(i)"
  - "B63"
  - "FN-CB-410a.2"
  - "NOTE-TO-FN-CB-410a.2"
  - "unknown"
- Do not guess paragraph numbers from the wording.
- Extract labelled sub-items separately. Do not merge paragraph 29(a), 29(b), 29(c), etc. into one record.
- section must be one primary section.
- report_sections must be a list of all report sections this item supports.
- obligation_type must be "shall", "should", or "may".
  - Use "shall" for mandatory wording or metric protocols that state "the entity shall..."
  - Use "should" for recommendations or "should" wording.
  - Use "may" for permitted optional disclosures or "may" wording.
- applicability must be one of:
  - commercial_banking
  - asset_management
  - insurance
  - general_financial_sector
  - not_financial_specific
- applies_to_banks must be true only for commercial_banking or general_financial_sector.
- related_paragraphs must be an empty list if no cross-reference is present.
- metric_type must be null if no specific metric is required.
- For FN-CB metrics, metric_type should contain the exact metric code where possible.

Return one JSON object only, with this shape:
{{
  "requirements": [
    {{
      "standard": "{standard}",
      "source_doc": "{source_doc}",
      "source_authority": "{source_authority}",
      "paragraph": "string",
      "section": "general_requirements|governance|strategy|risk_management|metrics_targets|industry_metrics|other",
      "report_sections": ["risk_management", "metrics_targets"],
      "obligation_type": "shall|should|may",
      "requirement_text": "string",
      "applies_to_banks": true,
      "applicability": "commercial_banking",
      "related_paragraphs": [],
      "metric_type": null,
      "page_start": {page_start},
      "page_end": {page_end}
    }}
  ]
}}

If the text contains no relevant disclosure requirements or guidance items, return:
{{"requirements": []}}

TEXT:
{text_chunk}
""".strip()


print("Prompts ready.")


Prompts ready.


In [12]:
# 11. GPT extraction function for one chunk

def fallback_applicability_without_keyword_mapping(raw_req: dict) -> str:
    if raw_req.get("standard") == "S2_IBG_CB":
        return "commercial_banking"

    if raw_req.get("applies_to_banks") is True:
        return "general_financial_sector"

    return "not_financial_specific"


def fallback_report_sections_without_keyword_mapping(raw_req: dict) -> list[str]:
    primary = raw_req.get("section", "other")
    if primary not in REPORT_SECTION_VALUES:
        primary = "other"

    sections = [primary]

    if raw_req.get("standard") == "S2_IBG_CB" and "industry_metrics" not in sections:
        sections.append("industry_metrics")

    return sorted(set(sections), key=REPORT_SECTION_VALUES.index)


def recover_paragraph_from_structure_only(raw_req: dict) -> str:
    current = normalise_paragraph_id(raw_req.get("paragraph", ""))

    if current != "unknown":
        return current

    text = str(raw_req.get("requirement_text", ""))

    match = re.search(r"\b(FN-CB-\d{3}[a-z]\.\d|FN-CB-\d{3}\.[A-Z])\b", text, flags=re.IGNORECASE)
    if match:
        return canonical_fn_cb_code(match.group(1))

    start_match = re.match(
        r"^\s*([A-Z]\d+[A-Z]?(?:\([a-z0-9]+\))*|\d{1,3}[A-Z]?(?:\([a-z0-9]+\))*)\b",
        text,
        flags=re.IGNORECASE,
    )

    if start_match:
        return normalise_paragraph_id(start_match.group(1))

    return "unknown"


def normalize_metric_type_from_text(raw_req: dict) -> str | None:
    value = raw_req.get("metric_type")

    text = " ".join([
        str(value or ""),
        str(raw_req.get("paragraph", "")),
        str(raw_req.get("requirement_text", "")),
    ])

    match = re.search(r"\b(FN-CB-\d{3}[a-z]\.\d|FN-CB-\d{3}\.[A-Z])\b", text, flags=re.IGNORECASE)
    if match:
        return canonical_fn_cb_code(match.group(1))

    if value in ["", "none", "None", "null", "NULL", "n/a", "N/A"]:
        return None

    return value


@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=20))
def extract_requirements_from_chunk(
    text_chunk: str,
    standard: str,
    source_doc: str,
    source_authority: str,
    page_start: int,
    page_end: int,
) -> list[dict]:
    result = call_extractor_llm_json(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=build_user_prompt(
            text_chunk=text_chunk,
            standard=standard,
            source_doc=source_doc,
            source_authority=source_authority,
            page_start=page_start,
            page_end=page_end,
        ),
    )

    if not isinstance(result, dict):
        raise ValueError("Extractor output must be a JSON object, but got: " + type(result).__name__)

    reqs = result.get("requirements", []) or []

    if not isinstance(reqs, list):
        raise ValueError(
            "Extractor output field `requirements` must be a list. Output preview:\n"
            + json.dumps(result, indent=2, ensure_ascii=False)[:3000]
        )

    unit_count = text_chunk.count("[PARAGRAPH_ID:")
    if unit_count >= 10 and len(reqs) / max(unit_count, 1) < 0.3:
        print(
            "WARNING: Low requirement-to-unit ratio. "
            f"requirements={len(reqs)}, paragraph_units={unit_count}. "
            "Manually inspect this chunk if it contains dense disclosure text."
        )

    cleaned = []

    for raw_req in reqs:
        if not isinstance(raw_req, dict):
            continue

        raw_req["standard"] = standard
        raw_req["source_doc"] = source_doc
        raw_req["source_authority"] = source_authority
        raw_req.setdefault("page_start", page_start)
        raw_req.setdefault("page_end", page_end)

        raw_req["paragraph"] = recover_paragraph_from_structure_only(raw_req)
        raw_req["metric_type"] = normalize_metric_type_from_text(raw_req)

        if not isinstance(raw_req.get("related_paragraphs"), list):
            raw_req["related_paragraphs"] = []

        if raw_req.get("applicability") not in APPLICABILITY_VALUES:
            raw_req["applicability"] = fallback_applicability_without_keyword_mapping(raw_req)

        if standard == "S2_IBG_CB":
            raw_req["applicability"] = "commercial_banking"

        raw_req["applies_to_banks"] = raw_req["applicability"] in [
            "commercial_banking",
            "general_financial_sector",
        ]

        if not isinstance(raw_req.get("report_sections"), list) or not raw_req.get("report_sections"):
            raw_req["report_sections"] = fallback_report_sections_without_keyword_mapping(raw_req)

        raw_req["report_sections"] = [
            s for s in raw_req["report_sections"]
            if s in REPORT_SECTION_VALUES
        ] or [raw_req.get("section", "other")]

        if raw_req.get("section") not in REPORT_SECTION_VALUES:
            raw_req["section"] = raw_req["report_sections"][0] if raw_req["report_sections"] else "other"

        allowed_fields = set(IFRSRequirement.model_fields.keys())
        raw_req = {k: v for k, v in raw_req.items() if k in allowed_fields}

        validated = IFRSRequirement.model_validate(raw_req)
        cleaned.append(validated.model_dump())

    return cleaned


print("Azure REST extraction function ready.")


Azure REST extraction function ready.


In [13]:
# 12. DEBUG: run extraction on only one paragraph-aware chunk first

# This cell calls the Azure OpenAI API.
# It is the best cell to debug model access, schema problems, and output quality.

pdf_info = available_pdfs[0]
pages = extract_pdf_pages(pdf_info["path"])

units = extract_paragraph_units_from_pages(
    pages=pages,
    source_doc=pdf_info["source_doc"],
    standard=pdf_info["standard"],
    source_authority=pdf_info["source_authority"],
)

chunks = make_chunks_from_units(units)

test_chunk_index = 0
test_chunk = chunks[test_chunk_index]

print("Testing on:")
print("Document:", pdf_info["source_doc"])
print("Standard:", pdf_info["standard"])
print("Authority:", pdf_info["source_authority"])
print("Chunk:", test_chunk_index + 1)
print("Pages:", test_chunk["page_start"], "-", test_chunk["page_end"])
print("Characters:", test_chunk["char_count"])
print("Unit count:", test_chunk["unit_count"])
print("Paragraphs preview:", test_chunk["paragraphs"][:20])

test_requirements = extract_requirements_from_chunk(
    text_chunk=test_chunk["text"],
    standard=pdf_info["standard"],
    source_doc=pdf_info["source_doc"],
    source_authority=pdf_info["source_authority"],
    page_start=test_chunk["page_start"],
    page_end=test_chunk["page_end"],
)

print(f"Requirements found: {len(test_requirements)}")

pd.DataFrame(test_requirements).head(20)


Testing on:
Document: IFRS S1 General Requirements
Standard: S1
Authority: core_standard
Chunk: 1
Pages: 1 - 9
Characters: 11974
Unit count: 47
Paragraphs preview: ['unknown', 'S1', '1', '5', '10', '11', '17', '20', '21', '25', '26', '28', '43', '45', '54', '54', '60', '64', '70', '72']
Requirements found: 10


,standard,source_doc,source_authority,paragraph,section,report_sections,obligation_type,requirement_text,applies_to_banks,applicability,related_paragraphs,metric_type,page_start,page_end
0,S1,IFRS S1 General Requirements,core_standard,3,general_requirements,[general_requirements],shall,This Standard requires an entity to disclose i...,False,not_financial_specific,[],None,7,7
1,S1,IFRS S1 General Requirements,core_standard,5,general_requirements,[general_requirements],shall,An entity shall apply this Standard in prepari...,False,not_financial_specific,[],None,7,7
2,S1,IFRS S1 General Requirements,core_standard,8,general_requirements,[general_requirements],may,An entity may apply IFRS Sustainability Disclo...,False,not_financial_specific,[],None,8,8
3,S1,IFRS S1 General Requirements,core_standard,11,general_requirements,[general_requirements],shall,A complete set of sustainability-related finan...,False,not_financial_specific,[],None,8,8
4,S1,IFRS S1 General Requirements,core_standard,12,general_requirements,[general_requirements],shall,To identify sustainability-related risks and o...,False,not_financial_specific,[B1],None,8,8
5,S1,IFRS S1 General Requirements,core_standard,13,general_requirements,[general_requirements],shall,"To achieve faithful representation, an entity ...",False,not_financial_specific,[],None,8,8
6,S1,IFRS S1 General Requirements,core_standard,15(a),general_requirements,[general_requirements],shall,Fair presentation also requires an entity: (a)...,False,not_financial_specific,[],None,8,9
7,S1,IFRS S1 General Requirements,core_standard,15(b),general_requirements,[general_requirements],shall,Fair presentation also requires an entity: (b)...,False,not_financial_specific,[],None,8,9
8,S1,IFRS S1 General Requirements,core_standard,17,general_requirements,[general_requirements],shall,An entity shall disclose material information ...,False,not_financial_specific,[],None,9,9
9,S1,IFRS S1 General Requirements,core_standard,19,general_requirements,[general_requirements],shall,"To identify and disclose material information,...",False,not_financial_specific,[B13],None,9,9


In [14]:
# 13. Inspect one extracted requirement in full

if not test_requirements:
    print("No requirements found in the test chunk.")
else:
    i = 0
    print(json.dumps(test_requirements[i], indent=2, ensure_ascii=False))


{
  "standard": "S1",
  "source_doc": "IFRS S1 General Requirements",
  "source_authority": "core_standard",
  "paragraph": "3",
  "section": "general_requirements",
  "report_sections": [
    "general_requirements"
  ],
  "obligation_type": "shall",
  "requirement_text": "This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash flows, its access to finance or cost of capital over the short, medium or long term.",
  "applies_to_banks": false,
  "applicability": "not_financial_specific",
  "related_paragraphs": [],
  "metric_type": null,
  "page_start": 7,
  "page_end": 7
}


In [15]:
# 14. Checkpoint helpers

def chunk_key(source_doc: str, chunk_index: int, page_start: int, page_end: int) -> str:
    return f"{source_doc}::chunk={chunk_index}::pages={page_start}-{page_end}"


def load_processed_chunk_keys(checkpoint_path: Path) -> set[str]:
    """Load chunk keys that were already processed."""
    if not checkpoint_path.exists():
        return set()

    keys = set()
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            keys.add(record["chunk_key"])

    return keys


def append_checkpoint_records(checkpoint_path: Path, key: str, requirements: list[dict]) -> None:
    """Append extracted requirements for one chunk to a JSONL checkpoint file."""
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for req in requirements:
            f.write(json.dumps({
                "chunk_key": key,
                "requirement": req,
            }, ensure_ascii=False) + "\n")


def load_checkpoint_requirements(checkpoint_path: Path) -> list[dict]:
    """Load all requirements from the JSONL checkpoint file."""
    if not checkpoint_path.exists():
        return []

    requirements = []
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            requirements.append(record["requirement"])

    return requirements


print("Checkpoint helpers ready.")


Checkpoint helpers ready.


In [16]:
# 15. FULL RUN: extract all requirements with checkpointing

# This cell calls the Azure OpenAI API many times.
# While debugging, set MAX_CHUNKS_PER_DOCUMENT = 1 or 2 in the config cell.
# When you are confident, set MAX_CHUNKS_PER_DOCUMENT = None and rerun.
#
# v3 uses a new checkpoint file:
#   ifrs_bank_requirements_raw_checkpoint_v3.jsonl
#
# This avoids mixing old page-level chunks with new paragraph-aware chunks.

processed_keys = load_processed_chunk_keys(OUTPUT_RAW_JSONL)
print(f"Already processed chunks in checkpoint: {len(processed_keys)}")

total_new_chunks = 0

for pdf_info in available_pdfs:
    pdf_path = pdf_info["path"]
    standard = pdf_info["standard"]
    source_doc = pdf_info["source_doc"]
    source_authority = pdf_info["source_authority"]

    print("\n" + "=" * 80)
    print(f"Processing: {source_doc}")
    print(f"Standard: {standard}")
    print(f"Authority: {source_authority}")
    print(f"File: {pdf_path}")

    pages = extract_pdf_pages(pdf_path)

    units = extract_paragraph_units_from_pages(
        pages=pages,
        source_doc=source_doc,
        standard=standard,
        source_authority=source_authority,
    )

    chunks = make_chunks_from_units(units)

    if MAX_CHUNKS_PER_DOCUMENT is not None:
        chunks = chunks[:MAX_CHUNKS_PER_DOCUMENT]

    print(f"Pages extracted: {len(pages)}")
    print(f"Paragraph-aware units extracted: {len(units)}")
    print(f"Chunks to process: {len(chunks)}")

    for idx, chunk in enumerate(chunks, start=1):
        key = chunk_key(source_doc, idx, chunk["page_start"], chunk["page_end"])

        if key in processed_keys:
            print(f"Skipping already processed chunk {idx}/{len(chunks)} pages {chunk['page_start']}-{chunk['page_end']}")
            continue

        print(
            f"Extracting chunk {idx}/{len(chunks)} "
            f"pages {chunk['page_start']}-{chunk['page_end']} "
            f"units={chunk['unit_count']}"
        )

        try:
            reqs = extract_requirements_from_chunk(
                text_chunk=chunk["text"],
                standard=standard,
                source_doc=source_doc,
                source_authority=source_authority,
                page_start=chunk["page_start"],
                page_end=chunk["page_end"],
            )

            append_checkpoint_records(OUTPUT_RAW_JSONL, key, reqs)
            processed_keys.add(key)
            total_new_chunks += 1

            print(f"  Found {len(reqs)} requirements")

            time.sleep(0.5)

        except Exception as e:
            print(f"  ERROR on chunk {idx}: {type(e).__name__}: {e}")
            print("  You can fix the issue and rerun this cell; checkpointed chunks will be skipped.")
            raise

print("\nFull run complete.")
print("New chunks processed:", total_new_chunks)
print("Checkpoint file:", OUTPUT_RAW_JSONL)


Already processed chunks in checkpoint: 40

Processing: IFRS S1 General Requirements
Standard: S1
Authority: core_standard
File: gen_data\IFRS\ifrs_s1.pdf
Pages extracted: 48
Paragraph-aware units extracted: 244
Chunks to process: 10
Skipping already processed chunk 1/10 pages 1-9
Skipping already processed chunk 2/10 pages 9-13
Skipping already processed chunk 3/10 pages 13-18
Skipping already processed chunk 4/10 pages 18-22
Skipping already processed chunk 5/10 pages 22-27
Skipping already processed chunk 6/10 pages 27-31
Skipping already processed chunk 7/10 pages 31-34
Skipping already processed chunk 8/10 pages 34-39
Skipping already processed chunk 9/10 pages 39-43
Skipping already processed chunk 10/10 pages 43-48

Processing: IFRS S2 Climate-related Disclosures
Standard: S2
Authority: core_standard
File: gen_data\IFRS\ifrs_s2.pdf
Pages extracted: 48
Paragraph-aware units extracted: 169
Chunks to process: 10
Skipping already processed chunk 1/10 pages 1-8
Skipping already proce

In [17]:
# 16. Load raw checkpoint results

raw_requirements = load_checkpoint_requirements(OUTPUT_RAW_JSONL)

print("Raw extracted requirements:", len(raw_requirements))

if raw_requirements:
    raw_df = pd.DataFrame(raw_requirements)
    display(raw_df.head(20))
else:
    print("No checkpoint requirements found yet.")


Raw extracted requirements: 840


,standard,source_doc,source_authority,paragraph,section,report_sections,obligation_type,requirement_text,applies_to_banks,applicability,related_paragraphs,metric_type,page_start,page_end
0,S1,IFRS S1 General Requirements,core_standard,4,general_requirements,[general_requirements],should,The Standard should be read in the context of ...,False,not_financial_specific,[],NaN,5,7
1,S1,IFRS S1 General Requirements,core_standard,4,general_requirements,[general_requirements],shall,An entity shall apply this Standard in prepari...,False,not_financial_specific,[],NaN,5,7
2,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],may,An entity may apply IFRS Sustainability Disclo...,False,not_financial_specific,[],NaN,7,8
3,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,A complete set of sustainability-related finan...,False,not_financial_specific,[],NaN,7,8
4,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,To identify sustainability-related risks and o...,False,not_financial_specific,[B1],NaN,7,8
5,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,"To achieve faithful representation, an entity ...",False,not_financial_specific,[],NaN,7,8
6,S1,IFRS S1 General Requirements,core_standard,6(a),general_requirements,[general_requirements],shall,"to disclose information that is comparable, ve...",False,not_financial_specific,[],NaN,8,8
7,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,to disclose additional information if complian...,False,not_financial_specific,[],NaN,8,9
8,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,An entity shall disclose material information ...,False,not_financial_specific,[],NaN,8,9
9,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,"To identify and disclose material information,...",False,not_financial_specific,[B13],NaN,8,9


In [18]:
# 17. Deduplication, sorting and generic cleanup

def normalize_text_for_dedupe(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def text_hash(text: str) -> str:
    normalised = normalize_text_for_dedupe(text)
    return hashlib.md5(normalised.encode("utf-8")).hexdigest()


def deduplicate_requirements(requirements: list[dict]) -> list[dict]:
    seen = set()
    deduped = []

    for req in requirements:
        key = (
            req.get("standard"),
            req.get("source_authority"),
            text_hash(req.get("requirement_text", "")),
        )

        if key not in seen:
            seen.add(key)
            deduped.append(req)

    return deduped


def infer_related_paragraphs(text: str) -> list[str]:
    text = str(text)

    patterns = [
        r"paragraphs?\s+([A-Z]?\d+[A-Z]?(?:[–-][A-Z]?\d+[A-Z]?)?)",
        r"see\s+paragraphs?\s+([A-Z]?\d+[A-Z]?(?:[–-][A-Z]?\d+[A-Z]?)?)",
        r"\b(IFRS\s+S[12]\s+paragraph\s+[A-Z]?\d+[A-Z]?)\b",
        r"\b(FN-CB-\d{3}[a-z]\.\d|FN-CB-\d{3}\.[A-Z])\b",
    ]

    found = []
    for pattern in patterns:
        found.extend(re.findall(pattern, text, flags=re.IGNORECASE))

    out = []
    for item in found:
        if isinstance(item, tuple):
            item = item[0]

        item = normalise_paragraph_id(str(item).strip())

        if item and item != "unknown" and item not in out:
            out.append(item)

    return out


def postprocess_requirement(req: dict) -> dict:
    req = dict(req)

    req.setdefault("source_authority", "industry_guidance" if req.get("standard") == "S2_IBG_CB" else "core_standard")
    req.setdefault("source_doc", "")
    req.setdefault("section", "other")
    req.setdefault("obligation_type", "shall")
    req.setdefault("related_paragraphs", [])
    req.setdefault("metric_type", None)
    req.setdefault("page_start", 0)
    req.setdefault("page_end", 0)

    req["paragraph"] = recover_paragraph_from_structure_only(req)
    req["metric_type"] = normalize_metric_type_from_text(req)

    if not isinstance(req.get("related_paragraphs"), list):
        req["related_paragraphs"] = []

    if not req["related_paragraphs"]:
        req["related_paragraphs"] = infer_related_paragraphs(req.get("requirement_text", ""))
    else:
        req["related_paragraphs"] = [
            normalise_paragraph_id(x)
            for x in req["related_paragraphs"]
            if normalise_paragraph_id(x) != "unknown"
        ]

    if req.get("applicability") not in APPLICABILITY_VALUES:
        req["applicability"] = fallback_applicability_without_keyword_mapping(req)

    if req.get("standard") == "S2_IBG_CB":
        req["source_authority"] = "industry_guidance"
        req["applicability"] = "commercial_banking"

    req["applies_to_banks"] = req["applicability"] in ["commercial_banking", "general_financial_sector"]

    report_sections = req.get("report_sections")
    valid_report_sections = []

    if isinstance(report_sections, list):
        valid_report_sections = [s for s in report_sections if s in REPORT_SECTION_VALUES]

    if not valid_report_sections:
        valid_report_sections = fallback_report_sections_without_keyword_mapping(req)

    req["report_sections"] = sorted(set(valid_report_sections), key=REPORT_SECTION_VALUES.index)

    if req.get("section") not in REPORT_SECTION_VALUES:
        req["section"] = req["report_sections"][0] if req["report_sections"] else "other"

    allowed_fields = set(IFRSRequirement.model_fields.keys())
    req = {k: v for k, v in req.items() if k in allowed_fields}

    validated = IFRSRequirement.model_validate(req)
    return validated.model_dump()


def postprocess_requirements(requirements: list[dict]) -> list[dict]:
    return [postprocess_requirement(r) for r in requirements]


def sort_requirements(requirements: list[dict]) -> list[dict]:
    section_order = {
        "general_requirements": 0,
        "governance": 1,
        "strategy": 2,
        "risk_management": 3,
        "metrics_targets": 4,
        "industry_metrics": 5,
        "other": 6,
    }

    obligation_order = {"shall": 0, "should": 1, "may": 2}
    authority_order = {"core_standard": 0, "industry_guidance": 1}

    return sorted(
        requirements,
        key=lambda r: (
            authority_order.get(r.get("source_authority", "core_standard"), 99),
            r.get("standard", ""),
            section_order.get(r.get("section", "other"), 99),
            obligation_order.get(r.get("obligation_type", "may"), 99),
            r.get("page_start", 999999),
            str(r.get("paragraph", "")),
        ),
    )


deduped_requirements = deduplicate_requirements(raw_requirements)
postprocessed_requirements = postprocess_requirements(deduped_requirements)
final_requirements = sort_requirements(postprocessed_requirements)

print("Raw requirements:", len(raw_requirements))
print("After deduplication:", len(deduped_requirements))
print("After post-processing:", len(final_requirements))

unknown_count = sum(1 for r in final_requirements if r.get("paragraph") == "unknown")
suspicious_count = sum(1 for r in final_requirements if is_suspicious_paragraph_id(r.get("paragraph")))

print("Remaining unknown paragraph values:", unknown_count)
print("Unknown paragraph rate:", f"{(unknown_count / max(len(final_requirements), 1)):.1%}")
print("Suspicious paragraph values:", suspicious_count)
print("Suspicious paragraph rate:", f"{(suspicious_count / max(len(final_requirements), 1)):.1%}")

print("\nThis final notebook tracks suspicious paragraph IDs separately instead of treating zero unknowns as success.")


Raw requirements: 840
After deduplication: 664
After post-processing: 664
Remaining unknown paragraph values: 10
Unknown paragraph rate: 1.5%
Suspicious paragraph values: 10
Suspicious paragraph rate: 1.5%

This final notebook tracks suspicious paragraph IDs separately instead of treating zero unknowns as success.


In [19]:
# 18. Validate final requirements against the schema

valid_requirements = []
invalid_requirements = []

for req in final_requirements:
    try:
        valid_req = IFRSRequirement.model_validate(req)
        valid_requirements.append(valid_req.model_dump())
    except ValidationError as e:
        invalid_requirements.append({
            "requirement": req,
            "errors": e.errors(),
        })

print("Valid requirements:", len(valid_requirements))
print("Invalid requirements:", len(invalid_requirements))

if invalid_requirements:
    print("\nExample invalid requirement:")
    print(json.dumps(invalid_requirements[0], indent=2, ensure_ascii=False))


Valid requirements: 664
Invalid requirements: 0


In [20]:
# 19. Save final JSON

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(valid_requirements, f, ensure_ascii=False, indent=2)

print(f"Saved final requirements JSON to: {OUTPUT_JSON.resolve()}")
print(f"Total saved requirements: {len(valid_requirements)}")


Saved final requirements JSON to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\ifrs_bank_requirements.json
Total saved requirements: 664


In [21]:
# 20. Quality-control summaries

if not valid_requirements:
    print("No valid requirements to summarize yet.")
else:
    df = pd.DataFrame(valid_requirements)

    print("By section:")
    display(df["section"].value_counts().rename_axis("section").reset_index(name="count"))

    print("By report section coverage:")
    exploded = df.explode("report_sections")
    display(exploded["report_sections"].value_counts().rename_axis("report_section").reset_index(name="count"))

    print("By standard:")
    display(df["standard"].value_counts().rename_axis("standard").reset_index(name="count"))

    print("By source authority:")
    display(df["source_authority"].value_counts().rename_axis("source_authority").reset_index(name="count"))

    print("By obligation type:")
    display(df["obligation_type"].value_counts().rename_axis("obligation_type").reset_index(name="count"))

    print("By applicability:")
    display(df["applicability"].value_counts().rename_axis("applicability").reset_index(name="count"))

    print("Bank-specific requirements:")
    display(df["applies_to_banks"].value_counts().rename_axis("applies_to_banks").reset_index(name="count"))

    unknown_df = df[df["paragraph"] == "unknown"]
    print(f"Unknown paragraph count: {len(unknown_df)} / {len(df)} ({len(unknown_df)/max(len(df),1):.1%})")
    display(unknown_df[["standard", "section", "requirement_text", "page_start", "page_end"]].head(20))

    display(df.head(20))


By section:


,section,count
0,metrics_targets,246
1,general_requirements,219
2,strategy,113
3,risk_management,46
4,governance,30
5,industry_metrics,10


By report section coverage:


,report_section,count
0,metrics_targets,308
1,general_requirements,228
2,strategy,164
3,risk_management,106
4,governance,48
5,industry_metrics,37


By standard:


...

Bank-specific requirements:


,applies_to_banks,count
0,False,592
1,True,72


Unknown paragraph count: 10 / 664 (1.5%)


,standard,section,requirement_text,page_start,page_end
336,S2,strategy,"Specifically, an entity shall disclose informa...",7,8
337,S2,strategy,"Specifically, an entity shall disclose informa...",7,8
338,S2,strategy,"Specifically, an entity shall disclose informa...",7,8
339,S2,strategy,"Specifically, an entity shall disclose informa...",7,8
340,S2,strategy,"Specifically, an entity shall disclose informa...",7,8
341,S2,strategy,An entity shall disclose information that enab...,7,8
342,S2,strategy,"Specifically, the entity shall: (a) describe c...",7,8
343,S2,strategy,"Specifically, the entity shall: (b) explain, f...",7,8
344,S2,strategy,"Specifically, the entity shall: (c) specify, f...",7,8
345,S2,strategy,"Specifically, the entity shall: (d) explain ho...",7,8


,standard,source_doc,source_authority,paragraph,section,report_sections,obligation_type,requirement_text,applies_to_banks,applicability,related_paragraphs,metric_type,page_start,page_end
0,S1,IFRS S1 General Requirements,core_standard,4,general_requirements,[general_requirements],shall,An entity shall apply this Standard in prepari...,False,not_financial_specific,[],NaN,5,7
1,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,A complete set of sustainability-related finan...,False,not_financial_specific,[],NaN,7,8
2,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,To identify sustainability-related risks and o...,False,not_financial_specific,[B1],NaN,7,8
3,S1,IFRS S1 General Requirements,core_standard,6,general_requirements,[general_requirements],shall,"To achieve faithful representation, an entity ...",False,not_financial_specific,[],NaN,7,8
4,S1,IFRS S1 General Requirements,core_standard,6(a),general_requirements,[general_requirements],shall,"to disclose information that is comparable, ve...",False,not_financial_specific,[],NaN,8,8
5,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,to disclose additional information if complian...,False,not_financial_specific,[],NaN,8,9
6,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,An entity shall disclose material information ...,False,not_financial_specific,[],NaN,8,9
7,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,"To identify and disclose material information,...",False,not_financial_specific,[B13],NaN,8,9
8,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,An entity’s sustainability-related financial d...,False,not_financial_specific,[B38],NaN,8,9
9,S1,IFRS S1 General Requirements,core_standard,6(b),general_requirements,[general_requirements],shall,An entity shall provide information in a manne...,False,not_financial_specific,[],NaN,8,9


In [22]:
# 20B. Critical requirements coverage check

def canonical_search_id(value: str) -> str:
    value = str(value or "").strip()
    if re.search(r"FN-CB-", value, flags=re.IGNORECASE):
        return canonical_fn_cb_code(value)
    return normalise_paragraph_id(value)


def requirement_contains_identifier(req: dict, identifier: str) -> tuple[bool, str]:
    identifier = canonical_search_id(identifier)
    identifier_lower = identifier.lower()

    paragraph = canonical_search_id(req.get("paragraph", ""))
    metric_type = canonical_search_id(req.get("metric_type", ""))
    related = [canonical_search_id(x) for x in req.get("related_paragraphs", [])]
    text = str(req.get("requirement_text", "")).lower()

    if paragraph.lower() == identifier_lower:
        return True, "paragraph_exact"

    if metric_type.lower() == identifier_lower:
        return True, "metric_type"

    if identifier_lower in [x.lower() for x in related]:
        return True, "related_paragraphs"

    if identifier_lower in text:
        return True, "requirement_text"

    if re.match(r"^\d{1,3}[A-Z]?$", identifier):
        if paragraph.lower().startswith(identifier_lower + "("):
            return True, "paragraph_subitem"

    return False, "missing"


CRITICAL_REQUIREMENTS = [
    ("S2", "29", "Cross-industry climate metric categories"),
    ("S2", "29A", "Scope 3 Category 15 / financed emissions limitation, if using amended IFRS S2"),
    ("S2", "29B", "Derivative relief disclosure, if using amended IFRS S2"),
    ("S2", "29C", "Category 15 total and financed emissions subtotal, if using amended IFRS S2"),
    ("S2", "B62", "Commercial banking financed-emissions application guidance"),
    ("S2", "B62A", "Commercial banking gross exposure / disaggregation application guidance"),
    ("S1", "17", "Material sustainability-related information"),
    ("S1", "19", "Apply materiality guidance B13-B37"),
    ("S1", "54", "Sources of guidance"),
    ("S1", "55", "SASB disclosure topics consideration"),
    ("S1", "57", "Sources for applicable disclosure requirements"),
    ("S1", "58", "SASB metrics consideration"),
    ("S2_IBG_CB", "FN-CB-410a.2", "ESG factors in credit analysis"),
    ("S2_IBG_CB", "FN-CB-000.A", "Checking and savings accounts activity metric"),
    ("S2_IBG_CB", "FN-CB-000.B", "Loans activity metric"),
]

coverage_rows = []

for standard, identifier, description in CRITICAL_REQUIREMENTS:
    matches = []
    match_sources = Counter()

    for r in valid_requirements:
        if r.get("standard") != standard:
            continue

        ok, source = requirement_contains_identifier(r, identifier)
        if ok:
            matches.append(r)
            match_sources[source] += 1

    coverage_rows.append({
        "standard": standard,
        "identifier": identifier,
        "description": description,
        "present": len(matches) > 0,
        "match_count": len(matches),
        "match_sources": dict(match_sources),
        "example_paragraphs": ", ".join(sorted({str(m.get("paragraph")) for m in matches})[:8]),
    })

critical_coverage_df = pd.DataFrame(coverage_rows)

print("Critical coverage check:")
display(critical_coverage_df)

missing_critical = critical_coverage_df[critical_coverage_df["present"] == False]

if len(missing_critical) > 0:
    print("\nWARNING: Some critical identifiers were not found anywhere.")
    print("This may be normal if your source PDF is older and does not contain amended paragraphs 29A/29B/29C.")
    print("Manually verify missing rows before using the KB as final.")
else:
    print("\nAll critical identifiers were found by paragraph, metric_type, related_paragraphs, or requirement_text.")


Critical coverage check:


,standard,identifier,description,present,match_count,match_sources,example_paragraphs
0,S2,29,Cross-industry climate metric categories,True,64,"{'requirement_text': 43, 'related_paragraphs':...","12(i)(7)(ii), 14(a), 14(a)(i)(1), 16(ii)(g)(i)..."
1,S2,29A,Scope 3 Category 15 / financed emissions limit...,True,3,"{'related_paragraphs': 2, 'paragraph_exact': 1}","16(ii)(g)(i), 29A"
2,S2,29B,"Derivative relief disclosure, if using amended...",False,0,{},
3,S2,29C,Category 15 total and financed emissions subto...,True,2,{'related_paragraphs': 2},"44(b), C6(b)"
4,S2,B62,Commercial banking financed-emissions applicat...,True,6,"{'related_paragraphs': 3, 'requirement_text': 3}","38(b)(ii)(c), 44(c), B37, B62A(a)(ii)(2), C6(c)"
5,S2,B62A,Commercial banking gross exposure / disaggrega...,True,4,"{'related_paragraphs': 2, 'requirement_text': 2}","38(b)(ii)(c), 44(c), B62A(a)(ii)(2), C6(c)"
6,S1,17,Material sustainability-related information,False,0,{},
7,S1,19,Apply materiality guidance B13-B37,False,0,{},
8,S1,54,Sources of guidance,True,2,"{'paragraph_exact': 1, 'related_paragraphs': 1}","24(b), 54"
9,S1,55,SASB disclosure topics consideration,True,3,"{'paragraph_subitem': 1, 'requirement_text': 2}","20(b)(ii)(a), 24(b), 55(a)"



This may be normal if your source PDF is older and does not contain amended paragraphs 29A/29B/29C.
Manually verify missing rows before using the KB as final.


In [23]:
# 20C. Suspicious paragraph QA

def build_suspicious_paragraph_report(requirements: list[dict]) -> pd.DataFrame:
    rows = []

    for r in requirements:
        paragraph = str(r.get("paragraph", ""))

        reasons = []

        if paragraph == "unknown":
            reasons.append("unknown")

        if paragraph.count("(") > MAX_PARAGRAPH_SUBLEVELS:
            reasons.append("too_many_sublevels")

        if "gaap" in paragraph.lower():
            reasons.append("contains_gaap_token")

        if len(paragraph) > 45:
            reasons.append("too_long")

        if is_suspicious_paragraph_id(paragraph):
            reasons.append("invalid_pattern")

        if reasons:
            rows.append({
                "standard": r.get("standard"),
                "paragraph": paragraph,
                "reasons": ";".join(sorted(set(reasons))),
                "section": r.get("section"),
                "report_sections": ";".join(r.get("report_sections", [])),
                "page_start": r.get("page_start"),
                "page_end": r.get("page_end"),
                "requirement_text_preview": str(r.get("requirement_text", ""))[:240],
            })

    return pd.DataFrame(rows)


suspicious_paragraph_df = build_suspicious_paragraph_report(valid_requirements)

print("Suspicious paragraph rows:", len(suspicious_paragraph_df))

if len(suspicious_paragraph_df) > 0:
    display(suspicious_paragraph_df.head(50))
else:
    print("No suspicious paragraph IDs detected.")

paragraph_counts = pd.Series([r.get("paragraph") for r in valid_requirements]).value_counts()
repeated_paragraphs = paragraph_counts[paragraph_counts >= 12].reset_index()
repeated_paragraphs.columns = ["paragraph", "count"]

print("\nHighly repeated paragraph IDs:")
display(repeated_paragraphs.head(30))


Suspicious paragraph rows: 10


,standard,paragraph,reasons,section,report_sections,page_start,page_end,requirement_text_preview
0,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, an entity shall disclose informa..."
1,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, an entity shall disclose informa..."
2,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, an entity shall disclose informa..."
3,S2,unknown,invalid_pattern;unknown,strategy,strategy;metrics_targets,7,8,"Specifically, an entity shall disclose informa..."
4,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, an entity shall disclose informa..."
5,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,An entity shall disclose information that enab...
6,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, the entity shall: (a) describe c..."
7,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, the entity shall: (b) explain, f..."
8,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, the entity shall: (c) specify, f..."
9,S2,unknown,invalid_pattern;unknown,strategy,strategy,7,8,"Specifically, the entity shall: (d) explain ho..."



Highly repeated paragraph IDs:


,paragraph,count
0,16(ii)(g)(i),26
1,14(vi)(b)(i),19
2,30,12
3,40(b)(ii)(c),12


In [24]:
# 21. Inspect requirements by report section

REPORT_SECTION_TO_INSPECT = "risk_management"

section_reqs = [
    r for r in valid_requirements
    if REPORT_SECTION_TO_INSPECT in r.get("report_sections", [])
]

print(f"Report section: {REPORT_SECTION_TO_INSPECT}")
print(f"Requirements / guidance items supporting this section: {len(section_reqs)}")

for r in section_reqs[:15]:
    print("\n" + "-" * 100)
    print(
        f"{r['standard']} ¶{r['paragraph']} | "
        f"authority={r['source_authority']} | "
        f"primary_section={r['section']} | "
        f"report_sections={r['report_sections']} | "
        f"obligation={r['obligation_type']} | "
        f"applicability={r['applicability']}"
    )
    print(r["requirement_text"][:1200])


Report section: risk_management
Requirements / guidance items supporting this section: 106

----------------------------------------------------------------------------------------------------
S1 ¶21(b)(i) | authority=core_standard | primary_section=general_requirements | report_sections=['general_requirements', 'governance', 'strategy', 'risk_management', 'metrics_targets'] | obligation=shall | applicability=not_financial_specific
An entity shall provide information in a manner that enables users of general purpose financial reports to understand the following types of connections: (b) the connections between disclosures provided by the entity: within its sustainability-related financial disclosures—such (i) as connections between disclosures on governance, strategy, risk management and metrics and targets;

----------------------------------------------------------------------------------------------------
S1 ¶6(b)(i) | authority=core_standard | primary_section=general_requirements |

In [25]:
# 22. Generate JSON-backed requirements_kb.py

def write_requirements_kb(requirements: list[dict], output_path: Path, data_path: Path) -> None:
    grouped_by_primary_section = defaultdict(list)
    grouped_by_report_section = defaultdict(list)

    for req in requirements:
        grouped_by_primary_section[req["section"]].append(req)

        for report_section in req.get("report_sections", []):
            grouped_by_report_section[report_section].append(req)

    data = {
        "metadata": {
            "generated_by": "01_extract_ifrs_requirements_S1_S2_CommercialBanks_AZURE_REST_FINAL.ipynb",
            "sources": [
                "IFRS S1 General Requirements",
                "IFRS S2 Climate-related Disclosures",
                "IFRS S2 Industry-based Guidance Volume 16 — Commercial Banks",
            ],
            "note": "core_standard items are IFRS S1/S2 core requirements; industry_guidance items are Commercial Banks guidance.",
        },
        "by_primary_section": dict(grouped_by_primary_section),
        "by_report_section": dict(grouped_by_report_section),
    }

    with open(data_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    kb_code = '''"""
Auto-generated IFRS S1/S2 + Commercial Banks requirements knowledge base.
Do not edit manually. Regenerate from the extraction notebook.

This module loads its data from requirements_kb_data.json.

Important:
- source_authority="core_standard" = IFRS S1/S2 core requirements
- source_authority="industry_guidance" = Commercial Banks industry-based guidance
"""

from __future__ import annotations

import json
from pathlib import Path


_DATA_PATH = Path(__file__).with_name("requirements_kb_data.json")

with open(_DATA_PATH, "r", encoding="utf-8") as f:
    _DATA = json.load(f)

REQUIREMENTS_BY_PRIMARY_SECTION = _DATA["by_primary_section"]
REQUIREMENTS_BY_REPORT_SECTION = _DATA["by_report_section"]

# Backward-compatible alias
REQUIREMENTS = REQUIREMENTS_BY_PRIMARY_SECTION


def _sort(reqs: list[dict]) -> list[dict]:
    section_order = {
        "general_requirements": 0,
        "governance": 1,
        "strategy": 2,
        "risk_management": 3,
        "metrics_targets": 4,
        "industry_metrics": 5,
        "other": 6,
    }

    authority_order = {
        "core_standard": 0,
        "industry_guidance": 1,
    }

    obligation_order = {
        "shall": 0,
        "should": 1,
        "may": 2,
    }

    return sorted(
        reqs,
        key=lambda r: (
            authority_order.get(r.get("source_authority", "core_standard"), 99),
            obligation_order.get(r.get("obligation_type", "shall"), 99),
            r.get("standard", ""),
            section_order.get(r.get("section", "other"), 99),
            r.get("page_start", 999999),
            str(r.get("paragraph", "")),
        )
    )


def _filter(
    reqs: list[dict],
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
    applicability: str | None = None,
) -> list[dict]:
    if standard:
        reqs = [r for r in reqs if r.get("standard") == standard]

    if source_authority:
        reqs = [r for r in reqs if r.get("source_authority") == source_authority]

    if mandatory_only:
        reqs = [r for r in reqs if r.get("obligation_type") == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r.get("applies_to_banks") is True]

    if applicability:
        reqs = [r for r in reqs if r.get("applicability") == applicability]

    return _sort(reqs)


def get_requirements(
    section: str,
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
    applicability: str | None = None,
) -> list[dict]:
    """Return requirements by primary extracted section."""
    reqs = REQUIREMENTS_BY_PRIMARY_SECTION.get(section, [])
    return _filter(reqs, standard, source_authority, mandatory_only, banks_only, applicability)


def get_requirements_for_report_section(
    report_section: str,
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
    applicability: str | None = None,
    include_guidance: bool = True,
) -> list[dict]:
    """Return requirements/guidance that support a generated report section."""
    reqs = REQUIREMENTS_BY_REPORT_SECTION.get(report_section, [])

    if not include_guidance:
        reqs = [r for r in reqs if r.get("source_authority") == "core_standard"]

    return _filter(reqs, standard, source_authority, mandatory_only, banks_only, applicability)


def get_core_requirements(report_section: str, mandatory_only: bool = True) -> list[dict]:
    """Return IFRS S1/S2 core standard requirements for a generated report section."""
    return get_requirements_for_report_section(
        report_section=report_section,
        source_authority="core_standard",
        mandatory_only=mandatory_only,
        include_guidance=False,
    )


def get_bank_guidance(report_section: str | None = None) -> list[dict]:
    """Return Commercial Banks industry guidance. If report_section is None, return all bank guidance."""
    if report_section is None:
        all_reqs = []
        for reqs in REQUIREMENTS_BY_REPORT_SECTION.values():
            all_reqs.extend(reqs)

        seen = set()
        out = []
        for r in all_reqs:
            key = (r.get("standard"), r.get("paragraph"), r.get("requirement_text", "")[:200])
            if key not in seen and r.get("standard") == "S2_IBG_CB":
                seen.add(key)
                out.append(r)

        return _sort(out)

    return get_requirements_for_report_section(
        report_section=report_section,
        standard="S2_IBG_CB",
        banks_only=True,
        include_guidance=True,
    )


def list_primary_sections() -> list[str]:
    return sorted(REQUIREMENTS_BY_PRIMARY_SECTION.keys())


def list_report_sections() -> list[str]:
    return sorted(REQUIREMENTS_BY_REPORT_SECTION.keys())


def count_requirements() -> int:
    all_reqs = []
    for reqs in REQUIREMENTS_BY_PRIMARY_SECTION.values():
        all_reqs.extend(reqs)
    return len(all_reqs)
'''

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(kb_code)


write_requirements_kb(valid_requirements, OUTPUT_KB, OUTPUT_KB_DATA)

print(f"Saved Python KB to: {OUTPUT_KB.resolve()}")
print(f"Saved KB data JSON to: {OUTPUT_KB_DATA.resolve()}")


Saved Python KB to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\requirements_kb.py
Saved KB data JSON to: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\requirements_kb_data.json


In [26]:
# 23. Test the generated KB without importing it

# This simulates how your report agents will query the KB.

requirements_by_report_section = defaultdict(list)

for req in valid_requirements:
    for report_section in req.get("report_sections", []):
        requirements_by_report_section[report_section].append(req)


def get_requirements_for_report_section_from_memory(
    report_section: str,
    standard: str | None = None,
    source_authority: str | None = None,
    mandatory_only: bool = False,
    banks_only: bool = False,
    applicability: str | None = None,
    include_guidance: bool = True,
) -> list[dict]:
    reqs = list(requirements_by_report_section.get(report_section, []))

    if not include_guidance:
        reqs = [r for r in reqs if r["source_authority"] == "core_standard"]

    if standard:
        reqs = [r for r in reqs if r["standard"] == standard]

    if source_authority:
        reqs = [r for r in reqs if r["source_authority"] == source_authority]

    if mandatory_only:
        reqs = [r for r in reqs if r["obligation_type"] == "shall"]

    if banks_only:
        reqs = [r for r in reqs if r["applies_to_banks"]]

    if applicability:
        reqs = [r for r in reqs if r["applicability"] == applicability]

    return sort_requirements(reqs)


test_core_governance = get_requirements_for_report_section_from_memory(
    report_section="governance",
    source_authority="core_standard",
    mandatory_only=True,
    include_guidance=False,
)

test_bank_risk_guidance = get_requirements_for_report_section_from_memory(
    report_section="risk_management",
    standard="S2_IBG_CB",
    banks_only=True,
)

print("Mandatory core governance requirements:", len(test_core_governance))
print("Commercial Banks risk-management guidance items:", len(test_bank_risk_guidance))

print("\nCore governance examples:")
for r in test_core_governance[:5]:
    print(f"- [{r['standard']} ¶{r['paragraph']}] {r['requirement_text'][:250]}...")

print("\nCommercial Banks risk-management guidance examples:")
for r in test_bank_risk_guidance[:5]:
    print(f"- [{r['standard']} ¶{r['paragraph']}] {r['requirement_text'][:250]}...")


Mandatory core governance requirements: 39
Commercial Banks risk-management guidance items: 31

Core governance examples:
- [S1 ¶21(b)(i)] An entity shall provide information in a manner that enables users of general purpose financial reports to understand the following types of connections: (b) the connections between disclosures provided by the entity: within its sustainability-relate...
- [S1 ¶6(b)(i)] within its sustainability-related financial disclosures —such as connections between disclosures on governance, strategy, risk management and metrics and targets; and...
- [S1 ¶B1] This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash flows, its access to finance or cost of capital over the short, medium or ...
- [S1 ¶32(ii)(b)] For example, if an entity integrates its oversight of sustainability-related risks and opportunities, the entity shall integrate the discl

In [27]:
# 24. Build a section-agent prompt using the extracted requirements

def build_section_prompt(section_name: str, payload: dict, requirements: list[dict]) -> str:
    core_reqs = [
        r for r in requirements
        if (
            section_name in r.get("report_sections", [])
            and r["source_authority"] == "core_standard"
            and r["obligation_type"] == "shall"
        )
    ]

    bank_guidance = [
        r for r in requirements
        if (
            section_name in r.get("report_sections", [])
            and r["standard"] == "S2_IBG_CB"
        )
    ]

    core_req_text = "\n".join(
        f"[{r['standard']} ¶{r['paragraph']}] {r['requirement_text']}"
        for r in core_reqs
    ) or "No core IFRS S1/S2 mandatory requirements were extracted for this section."

    bank_guidance_text = "\n".join(
        f"[{r['standard']} ¶{r['paragraph']}] {r['requirement_text']}"
        for r in bank_guidance
    ) or "No Commercial Banks industry guidance was extracted for this section."

    return f"""
You are writing the {section_name.replace("_", " ").title()} section of an IFRS S1/S2 sustainability report for a commercial bank.

CORE IFRS S1/S2 MANDATORY DISCLOSURE REQUIREMENTS:
You must address every applicable core requirement below.

{core_req_text}

COMMERCIAL BANKS INDUSTRY-BASED GUIDANCE:
Use this as banking-specific guidance to make the section relevant to commercial banking.
This guidance accompanies IFRS S2 and supports application for banks, but do not present it as creating separate IFRS requirements.

{bank_guidance_text}

BANK DATA:
{json.dumps(payload, indent=2, ensure_ascii=False)}

Instructions:
1. Write a professional sustainability disclosure section.
2. Use only the bank data provided.
3. Do not invent metrics.
4. If data is insufficient for a requirement or guidance item, explicitly state the missing data.
5. Reference IFRS paragraph numbers and FN-CB metric codes where relevant.
6. Keep core IFRS S1/S2 requirements separate from banking-specific guidance in your reasoning.
7. Treat `core_standard` items as mandatory compliance requirements.
8. Treat `industry_guidance` items as supporting guidance for banking relevance.
9. Keep the tone close to a real annual sustainability report.
""".strip()


# Example fake payload just to test prompt construction.
sample_governance_payload = {
    "bank_name": "Example Bank",
    "reporting_year": 2024,
    "board_oversight": {
        "board_climate_meetings": 6,
        "climate_reports_frequency": "semi-annual",
        "responsible_committee": "Risk Committee",
    },
    "management_role": {
        "executive_owner": "Chief Risk Officer",
        "climate_risk_team": "Enterprise Risk Management",
    },
}

governance_prompt = build_section_prompt(
    section_name="governance",
    payload=sample_governance_payload,
    requirements=valid_requirements,
)

print(governance_prompt[:4000])


You are writing the Governance section of an IFRS S1/S2 sustainability report for a commercial bank.

CORE IFRS S1/S2 MANDATORY DISCLOSURE REQUIREMENTS:
You must address every applicable core requirement below.

[S1 ¶21(b)(i)] An entity shall provide information in a manner that enables users of general purpose financial reports to understand the following types of connections: (b) the connections between disclosures provided by the entity: within its sustainability-related financial disclosures—such (i) as connections between disclosures on governance, strategy, risk management and metrics and targets;
[S1 ¶6(b)(i)] within its sustainability-related financial disclosures —such as connections between disclosures on governance, strategy, risk management and metrics and targets; and
[S1 ¶B1] This Standard requires an entity to disclose information about all sustainability-related risks and opportunities that could reasonably be expected to affect the entity’s cash flows, its access to fi

In [28]:
# 25. Optional: generate one report section with GPT
# This cell calls the same Azure extractor endpoint.
# Run it only after you are satisfied with the prompt preview above.
#
# Note: for final report writing, you may prefer your existing writer endpoint
# from the governance notebook. This cell is only a simple test.

def generate_section_draft(section_name: str, payload: dict, requirements: list[dict]) -> str:
    prompt = build_section_prompt(
        section_name=section_name,
        payload=payload,
        requirements=requirements,
    )

    data = _azure_chat_completion(
        url=AZURE_OPENAI_EXTRACTOR_URL,
        api_key=AZURE_OPENAI_EXTRACTOR_API_KEY,
        messages=[
            {
                "role": "system",
                "content": "You are an IFRS S1/S2 sustainability reporting specialist for banking.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        max_output_tokens=3000,
        use_max_completion_tokens=True,
        temperature=None,
        json_mode=False,
        request_label="GPT-5.2 section draft test",
    )

    return _extract_message_content(data)


# Uncomment to test:
# governance_draft = generate_section_draft(
#     section_name="governance",
#     payload=sample_governance_payload,
#     requirements=valid_requirements,
# )
#
# print(governance_draft)


In [29]:
# 26. Build coverage checklists for QA

def build_coverage_checklist(
    report_section: str,
    requirements: list[dict],
    include_guidance: bool = True,
) -> pd.DataFrame:
    section_reqs = []

    for r in requirements:
        if report_section not in r.get("report_sections", []):
            continue

        # Always include mandatory core IFRS requirements.
        if r["source_authority"] == "core_standard" and r["obligation_type"] == "shall":
            section_reqs.append(r)

        # Optionally include Commercial Banks industry guidance.
        elif include_guidance and r["standard"] == "S2_IBG_CB":
            section_reqs.append(r)

    rows = []

    for r in section_reqs:
        rows.append({
            "report_section": report_section,
            "standard": r["standard"],
            "source_authority": r["source_authority"],
            "paragraph": r["paragraph"],
            "source_ref": f"{r['standard']} ¶{r['paragraph']} p.{r['page_start']}-{r['page_end']}",
            "primary_section": r["section"],
            "all_report_sections": ";".join(r.get("report_sections", [])),
            "obligation_type": r["obligation_type"],
            "applicability": r["applicability"],
            "applies_to_banks": r["applies_to_banks"],
            "metric_type": r["metric_type"],
            "related_paragraphs": ";".join(r.get("related_paragraphs", [])),
            "requirement_text": r["requirement_text"],
            "covered_in_draft": None,
            "evidence_sentence": None,
            "missing_data": None,
            "qa_comment": None,
        })

    return pd.DataFrame(rows)


coverage_checklists = {}

for section_name in [
    "general_requirements",
    "governance",
    "strategy",
    "risk_management",
    "metrics_targets",
    "industry_metrics",
]:
    coverage_checklists[section_name] = build_coverage_checklist(
        report_section=section_name,
        requirements=valid_requirements,
        include_guidance=True,
    )

    print(section_name, "rows:", len(coverage_checklists[section_name]))

coverage_checklists["governance"].head(20)


general_requirements rows: 199
governance rows: 48
strategy rows: 159
risk_management rows: 102
metrics_targets rows: 293
industry_metrics rows: 37


,report_section,standard,source_authority,paragraph,source_ref,primary_section,all_report_sections,obligation_type,applicability,applies_to_banks,metric_type,related_paragraphs,requirement_text,covered_in_draft,evidence_sentence,missing_data,qa_comment
0,governance,S1,core_standard,21(b)(i),S1 ¶21(b)(i) p.9-9,general_requirements,general_requirements;governance;strategy;risk_...,shall,not_financial_specific,False,NaN,,An entity shall provide information in a manne...,None,None,None,None
1,governance,S1,core_standard,6(b)(i),S1 ¶6(b)(i) p.9-9,general_requirements,general_requirements;governance;strategy;risk_...,shall,not_financial_specific,False,NaN,,within its sustainability-related financial di...,None,None,None,None
2,governance,S1,core_standard,B1,S1 ¶B1 p.26-26,general_requirements,general_requirements;governance;strategy;risk_...,shall,not_financial_specific,False,NaN,3,This Standard requires an entity to disclose i...,None,None,None,None
3,governance,S1,core_standard,32(ii)(b),S1 ¶32(ii)(b) p.34-34,general_requirements,general_requirements;governance;strategy;risk_...,shall,not_financial_specific,False,NaN,,"For example, if an entity integrates its overs...",None,None,None,None
4,governance,S1,core_standard,42,S1 ¶42 p.43-45,general_requirements,general_requirements;governance;strategy;risk_...,shall,not_financial_specific,False,NaN,,"If an entity uses this transition relief, it s...",None,None,None,None
5,governance,S1,core_standard,25(a),S1 ¶25(a) p.10-10,governance,governance,shall,not_financial_specific,False,NaN,26,Unless another IFRS Sustainability Disclosure ...,None,None,None,None
6,governance,S1,core_standard,27(a),S1 ¶27(a) p.10-10,governance,governance,shall,not_financial_specific,False,NaN,,"To achieve this objective, an entity shall dis...",None,None,None,None
7,governance,S1,core_standard,27(a)(i),S1 ¶27(a)(i) p.10-10,governance,governance,shall,not_financial_specific,False,NaN,,"Specifically, the entity shall identify that b...",None,None,None,None
8,governance,S1,core_standard,8(a),S1 ¶8(a) p.10-10,governance,governance,shall,not_financial_specific,False,NaN,26;27,Unless another IFRS Sustainability Disclosure ...,None,None,None,None
9,governance,S1,core_standard,8(a),S1 ¶8(a) p.10-10,governance,governance,shall,not_financial_specific,False,NaN,,"To achieve this objective, an entity shall dis...",None,None,None,None


In [30]:
# 27. Save all checklist templates

saved_checklists = []

for section_name, checklist_df in coverage_checklists.items():
    path = CHECKLIST_DIR / f"{section_name}_coverage_checklist_template.csv"
    checklist_df.to_csv(path, index=False, encoding="utf-8-sig")
    saved_checklists.append(path)

print("Saved checklist templates:")
for path in saved_checklists:
    print("-", path.resolve())

# Optional combined checklist
combined_checklist = pd.concat(
    [df for df in coverage_checklists.values() if len(df) > 0],
    ignore_index=True,
)

COMBINED_CHECKLIST_PATH = CHECKLIST_DIR / "all_sections_coverage_checklist_template.csv"
combined_checklist.to_csv(COMBINED_CHECKLIST_PATH, index=False, encoding="utf-8-sig")

print("\nSaved combined checklist:")
print(COMBINED_CHECKLIST_PATH.resolve())
print("Combined rows:", len(combined_checklist))


Saved checklist templates:
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\general_requirements_coverage_checklist_template.csv
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\governance_coverage_checklist_template.csv
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\strategy_coverage_checklist_template.csv
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\risk_management_coverage_checklist_template.csv
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\metrics_targets_coverage_checklist_template.csv
- C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\requirements\coverage_checklists\industry_metrics_coverage_checklist_template.csv

Saved combined checklist:
C:\Users\BV426BP\Docum

## How to use this notebook in your pipeline

After you run the full extraction and cleanup, use these files in your project:

```text
data/requirements/ifrs_bank_requirements.json
data/requirements/requirements_kb.py
data/requirements/requirements_kb_data.json
data/requirements/coverage_checklists/
```

The old enrichment duplicate was removed. There is now only one final requirements JSON:

```text
ifrs_bank_requirements.json
```

Your section agents should receive:

1. The bank payload for the section
2. The mandatory IFRS S1/S2 core requirements for the same section
3. The Commercial Banks industry-based guidance where relevant
4. Instructions to satisfy each core requirement and use the industry guidance without treating it as a separate IFRS requirement

### Final notebook flow

```text
PDF page text
   ↓
detect visible paragraph/metric markers with regex
   ↓
create paragraph-aware units
   ↓
build chunks with [PARAGRAPH_ID: ...] headers
   ↓
ask GPT to extract each labelled sub-item separately
   ↓
validate with Pydantic
   ↓
normalise paragraph IDs and FN-CB codes
   ↓
run suspicious paragraph QA
   ↓
run critical coverage check
   ↓
write final JSON + JSON-backed KB + QA checklists
```

### Important checkpoint note

This final version uses a new checkpoint file:

```text
ifrs_bank_requirements_raw_checkpoint_final.jsonl
```

This avoids mixing old v1/v2/v3/v4 chunks with the final extraction logic.

### Required source PDFs

Put exactly these three PDFs in `data/standards/`:

```text
ifrs_s1.pdf
ifrs_s2.pdf
ifrs_s2_ibg_volume_16_commercial_banks.pdf
```

### Recommended run order

1. One-chunk extraction test
2. Full extraction
3. Deduplication + generic post-processing
4. Validation
5. Save final JSON
6. Run critical coverage check
7. Run suspicious paragraph QA
8. Generate KB
9. Generate all checklists

Do not use the old v1/v2/v3/v4 checkpoint if you want the final clean results.
